# Classificador leve INT8 para RF-02/03/04 (tampa ausente, mal rosqueada, deformidade, normal)

Segunda camada do stack: **aparência/classes**. Transfer learning com **MobileNetV3-Small** (leve, INT8-friendly). Roda no Colab agora desde que exista a pasta de dados.

## 1) Estrutura de dados esperada

```
/content/data/
  normal/
  cap_ausente/
  cap_mal_rosqueada/
  deformidade/
```

Monte as pastas com algumas dezenas de imagens por classe, ou substitua `DATA_ROOT` por onde estiverem suas imagens.

In [ ]:
import os, torch, torchvision
from torchvision import datasets, transforms, models

DATA_ROOT = "/content/data"
if not os.path.isdir(DATA_ROOT):
    raise SystemExit("Sem dados ainda: monte /content/data com normal/, cap_ausente/, cap_mal_rosqueada/, deformidade/")

tr = transforms.Compose([
    transforms.Resize((224,224)), transforms.RandomHorizontalFlip(),
    transforms.ToTensor(), transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
dset = datasets.ImageFolder(DATA_ROOT, transform=tr)
print("classes:", dset.classes, "| total:", len(dset))
train, val = torch.utils.data.random_split(dset, [int(0.8*len(dset)), len(dset)-int(0.8*len(dset))])
dl  = torch.utils.data.DataLoader(train, batch_size=16, shuffle=True, num_workers=2)
dlv = torch.utils.data.DataLoader(val,  batch_size=16, shuffle=False, num_workers=2)

## 2) Exemplos do dataset carregado (preview)

Mostra imagens de cada classe carregada, para conferir rótulos antes de treinar.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image

classes = dset.classes
fig, axes = plt.subplots(len(classes), 4, figsize=(12, 3*len(classes)))
if len(classes) == 1:
    axes = [axes]
for i, c in enumerate(classes):
    fs = sorted((Path(DATA_ROOT)/c).glob('*'))[:4]
    for j in range(4):
        ax = axes[i][j]
        if j < len(fs):
            ax.imshow(Image.open(fs[j])); ax.set_title(c, fontsize=9)
        ax.axis('off')
plt.tight_layout(); plt.show()
print("contagem por classe:", {c: len(list((Path(DATA_ROOT)/c).glob('*'))) for c in classes})

## 3) Modelo (MobileNetV3-Small, transfer learning)

In [ ]:
model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
model.classifier[3] = torch.nn.Linear(model.classifier[3].in_features, len(dset.classes))
dev = "cuda" if torch.cuda.is_available() else "cpu"
model.to(dev)

## 4) Treino

In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=1e-4)
lossf = torch.nn.CrossEntropyLoss()
model.train()
for ep in range(5):
    tot=0.0; cnt=0
    for x,y in dl:
        x,y = x.to(dev), y.to(dev)
        opt.zero_grad(); out=model(x); loss=lossf(out,y); loss.backward(); opt.step()
        tot += loss.item(); cnt += 1
    print(f"ep {ep} loss {tot/cnt:.4f}")

## 5) Avaliação (matriz de confusão + acurácia)

In [ ]:
from collections import Counter
model.eval()
conf = Counter(); total = 0; acertos = 0
with torch.no_grad():
    for x,y in dlv:
        x,y = x.to(dev), y.to(dev)
        pred = model(x).argmax(1)
        for p,t in zip(pred.tolist(), y.tolist()):
            conf[(classes[t], classes[p])] += 1
            total += 1; acertos += int(p == t)
print("acuracia:", round(acertos/total, 4))
for k,v in sorted(conf.items()):
    print(f"  {k[0]:>18s} -> {k[1]:<18s} {v}")

## 6) Exportar INT8 para o Pi 5 (passo seguinte)

Com o dataset próprio validando o gate RNF-02, exportamos o modelo para TFLite INT8 (via ONNX→TFLite) e rodamos no Pi 5. Este notebook cobre a validação Python; a exportação é etapa separada.